# Chapter 09: Complete Integration - Building the Full OpenEvolve System

> From a pile of independent modules to a unified evolutionary coding engine

## Pain Point

We've built all the key components in previous chapters:
- **Chapter 01**: Basic evolution loop
- **Chapter 02**: MAP-Elites diversity preservation
- **Chapter 03**: Island-based multi-population evolution
- **Chapter 04**: LLM-driven intelligent mutation
- **Chapter 05**: Cascade evaluation pipeline
- **Chapter 06**: Diff-based code evolution
- **Chapter 07**: Process-parallel execution
- **Chapter 08**: Checkpoint/resume mechanism

**But they're all separate pieces!** How do we wire them into a single, production-ready system that a user can invoke with one command?

This chapter builds the **Controller** (the brain that orchestrates everything) and a **high-level API** for easy usage.

## 9.1 System Architecture Overview

```mermaid
flowchart TB
    subgraph User["User Layer"]
        CLI["CLI Entry Point<br/>openevolve-run.py"]
        API["Python API<br/>run_evolution()"]
    end

    subgraph Controller["Controller Layer"]
        OE["OpenEvolve Controller<br/>Orchestrates everything"]
        CFG["Config System<br/>YAML + Defaults"]
    end

    subgraph Core["Core Components"]
        PS["PromptSampler<br/>Build LLM prompts"]
        LLM["LLM Ensemble<br/>Generate mutations"]
        EVAL["Cascade Evaluator<br/>Score programs"]
        DB["ProgramDatabase<br/>MAP-Elites + Islands"]
        DIFF["Diff Engine<br/>SEARCH/REPLACE"]
    end

    subgraph Infra["Infrastructure"]
        PAR["Process Pool<br/>Parallel execution"]
        CKP["Checkpoint Manager<br/>Save/resume"]
        TRACE["Evolution Tracer<br/>Track lineage"]
    end

    CLI --> OE
    API --> OE
    CFG --> OE
    OE --> PS --> LLM --> DIFF
    OE --> EVAL
    OE --> DB
    OE --> PAR
    OE --> CKP
    OE --> TRACE
```

### Data Flow in One Iteration

```
1. Controller asks Database for parent programs (sampling)
2. PromptSampler builds a prompt with parent code + evolution history
3. LLM Ensemble generates a mutation (diff or full rewrite)
4. Diff Engine applies the changes to produce child code
5. Cascade Evaluator scores the child (3-stage pipeline)
6. Database stores the child in the MAP-Elites grid
7. Checkpoint Manager periodically saves state
8. Repeat until convergence or max iterations
```

## 9.2 Configuration System

A real system needs flexible configuration. OpenEvolve uses a **hierarchical config** with YAML files and sensible defaults.

### Config Hierarchy

$$\text{Final Config} = \text{Defaults} \leftarrow \text{YAML File} \leftarrow \text{CLI Arguments}$$

**Concrete Example:**

> Default: `max_iterations = 10000`
> YAML file sets: `max_iterations: 500`
> CLI argument: `--iterations 100`
> Final value: `max_iterations = 100` (CLI wins)

**Life Analogy:** Like cooking with a recipe book (defaults), a family recipe note (YAML), and last-minute taste adjustments (CLI) — the most recent override wins.

In [ ]:
"""Configuration system for our OpenEvolve implementation"""

from dataclasses import dataclass, field
from typing import List, Optional, Dict, Any
import os

@dataclass
class LLMModelConfig:
    """Configuration for a single LLM model"""
    name: str = "gpt-4"
    weight: float = 1.0
    api_key: Optional[str] = None
    api_base: str = "https://api.openai.com/v1"
    temperature: float = 0.7
    max_tokens: int = 4096
    timeout: int = 60

@dataclass
class DatabaseConfig:
    """Configuration for program database"""
    num_islands: int = 5
    feature_bins: int = 10
    migration_interval: int = 50
    migration_rate: float = 0.1
    feature_dimensions: List[str] = field(
        default_factory=lambda: ["complexity", "diversity"]
    )

@dataclass
class EvaluatorConfig:
    """Configuration for evaluation pipeline"""
    cascade_evaluation: bool = True
    cascade_thresholds: List[float] = field(
        default_factory=lambda: [0.5, 0.75, 0.9]
    )
    timeout: int = 300

@dataclass
class Config:
    """Master configuration for OpenEvolve"""

    # General settings
    max_iterations: int = 10000
    checkpoint_interval: int = 100
    random_seed: Optional[int] = 42

    # Component configs
    llm: LLMModelConfig = field(default_factory=LLMModelConfig)
    database: DatabaseConfig = field(default_factory=DatabaseConfig)
    evaluator: EvaluatorConfig = field(default_factory=EvaluatorConfig)

    # Evolution settings
    diff_based_evolution: bool = True

    # Early stopping
    early_stopping_patience: Optional[int] = None
    convergence_threshold: float = 0.001
    early_stopping_metric: str = "combined_score"

    @classmethod
    def from_dict(cls, config_dict: Dict[str, Any]) -> "Config":
        """Create config from dictionary (e.g., from YAML)"""
        config = cls()
        for key, value in config_dict.items():
            if key == "llm" and isinstance(value, dict):
                config.llm = LLMModelConfig(**value)
            elif key == "database" and isinstance(value, dict):
                config.database = DatabaseConfig(**value)
            elif key == "evaluator" and isinstance(value, dict):
                config.evaluator = EvaluatorConfig(**value)
            elif hasattr(config, key):
                setattr(config, key, value)
        return config

    @classmethod
    def from_yaml(cls, path: str) -> "Config":
        """Load config from YAML file"""
        import yaml
        with open(path, 'r') as f:
            config_dict = yaml.safe_load(f)
        return cls.from_dict(config_dict)

# Test it
config = Config()
print(f"Default config:")
print(f"  max_iterations: {config.max_iterations}")
print(f"  checkpoint_interval: {config.checkpoint_interval}")
print(f"  num_islands: {config.database.num_islands}")
print(f"  cascade_thresholds: {config.evaluator.cascade_thresholds}")
print(f"  diff_based_evolution: {config.diff_based_evolution}")

# Override from dict (simulating YAML load)
custom = Config.from_dict({
    "max_iterations": 500,
    "database": {"num_islands": 3, "migration_interval": 25},
    "early_stopping_patience": 50
})
print(f"\nCustom config:")
print(f"  max_iterations: {custom.max_iterations}")
print(f"  num_islands: {custom.database.num_islands}")
print(f"  early_stopping_patience: {custom.early_stopping_patience}")

## 9.3 Early Stopping

### Why Early Stopping?

Without it, evolution runs for all `max_iterations` even if no improvement is happening. This wastes compute (and money for LLM API calls).

### Two Modes

**Mode 1: Patience-based** (patience > 0)

$$\text{Stop if } \Delta_{\text{best}} < \epsilon \text{ for } P \text{ consecutive iterations}$$

Where:
- $\Delta_{\text{best}}$ = improvement in best score
- $\epsilon$ = `convergence_threshold` (e.g., 0.001)
- $P$ = `patience` (e.g., 50 iterations)

**Concrete Example:**
> patience = 50, threshold = 0.001
> At iteration 200, best score = 0.85
> For the next 50 iterations (201-250), best score stays at 0.852
> $\Delta = 0.852 - 0.85 = 0.002 > 0.001$, so we keep going
> But if $\Delta < 0.001$ for 50 straight iterations → stop!
>
> **Life Analogy:** Like fishing — if you haven’t caught anything for an hour (patience), pack up and go home.

**Mode 2: Target-based** (patience < 0 or patience = None with target_score)

$$\text{Stop if } \text{score} \geq \text{target\_score}$$

> Set target_score = 0.95, and evolution stops as soon as any program reaches 0.95.
>
> **Life Analogy:** Like a thermostat — heater turns off when room reaches target temperature.

In [ ]:
"""Early stopping logic"""

class EarlyStopping:
    """Monitors evolution progress and triggers early stopping"""

    def __init__(self, patience=None, threshold=0.001, metric="combined_score"):
        self.patience = patience        # None = disabled
        self.threshold = threshold
        self.metric = metric
        self.best_score = float('-inf')
        self.iterations_without_improvement = 0
        self.triggered = False

    def check(self, metrics, target_score=None):
        """
        Check if early stopping should trigger.

        Returns: (should_stop, reason)
        """
        if self.patience is None and target_score is None:
            return False, ""

        # Extract score from metrics
        if self.metric in metrics:
            current_score = metrics[self.metric]
        else:
            # Fallback: average of numeric metrics
            numeric = [v for v in metrics.values()
                      if isinstance(v, (int, float)) and not isinstance(v, bool)]
            current_score = sum(numeric) / len(numeric) if numeric else 0.0

        # Mode 2: Target-based stopping
        if target_score is not None and current_score >= target_score:
            self.triggered = True
            return True, f"Target score {target_score} reached (current: {current_score:.4f})"

        # Mode 1: Patience-based stopping
        if self.patience is not None and self.patience > 0:
            improvement = current_score - self.best_score

            if improvement > self.threshold:
                # Real improvement found
                self.best_score = current_score
                self.iterations_without_improvement = 0
            else:
                self.iterations_without_improvement += 1

            if self.iterations_without_improvement >= self.patience:
                self.triggered = True
                return True, (
                    f"No improvement for {self.patience} iterations "
                    f"(best: {self.best_score:.4f}, threshold: {self.threshold})"
                )
        else:
            # Update best score even without patience-based stopping
            if current_score > self.best_score:
                self.best_score = current_score

        return False, ""

# Demo: Patience-based early stopping
print("=== Demo: Patience-based Early Stopping ===")
es = EarlyStopping(patience=5, threshold=0.01)

scores = [0.3, 0.5, 0.65, 0.72, 0.78, 0.79, 0.791, 0.792, 0.793, 0.794, 0.795]
for i, score in enumerate(scores):
    should_stop, reason = es.check({"combined_score": score})
    status = "STOP!" if should_stop else f"stale={es.iterations_without_improvement}"
    print(f"  Iter {i}: score={score:.3f}, best={es.best_score:.3f}, {status}")
    if should_stop:
        print(f"  >> {reason}")
        break

print()

# Demo: Target-based early stopping
print("=== Demo: Target-based Early Stopping ===")
es2 = EarlyStopping()  # No patience needed
scores2 = [0.3, 0.5, 0.7, 0.85, 0.92, 0.96]
for i, score in enumerate(scores2):
    should_stop, reason = es2.check({"combined_score": score}, target_score=0.95)
    print(f"  Iter {i}: score={score:.3f}, stop={should_stop}")
    if should_stop:
        print(f"  >> {reason}")
        break

## 9.4 The Controller: Tying Everything Together

The Controller is the “brain” of OpenEvolve. It:

1. **Initializes** all components from config
2. **Evaluates** the initial program as the seed
3. **Runs** the evolution loop (potentially in parallel)
4. **Saves checkpoints** periodically
5. **Checks early stopping** after each iteration
6. **Returns** the best program found

### Controller Lifecycle

```mermaid
flowchart TD
    A["__init__()"] --> B["Load config & create components"]
    B --> C["run()"]
    C --> D["Evaluate initial program"]
    D --> E["Add to database as seed"]
    E --> F{"Iteration < max?"}
    F -->|Yes| G["Sample parents from DB"]
    G --> H["Build prompt"]
    H --> I["LLM generates mutation"]
    I --> J["Apply diff / rewrite"]
    J --> K["Cascade evaluate child"]
    K --> L["Store in database"]
    L --> M{"Checkpoint interval?"}
    M -->|Yes| N["Save checkpoint"]
    M -->|No| O{"Early stopping?"}
    N --> O
    O -->|No| F
    O -->|Yes| P["Save final state"]
    F -->|No| P
    P --> Q["Return best program"]
```

In [ ]:
"""
The OpenEvolve Controller - Complete Integration

This ties together everything we built in chapters 01-08.
"""

import random
import time
import uuid
import copy
import os

class OpenEvolveController:
    """
    Main controller that orchestrates the entire evolution process.

    This is the 'brain' of OpenEvolve - it coordinates:
    - Database (MAP-Elites + Islands) from Chapter 02-03
    - LLM mutation from Chapter 04
    - Cascade evaluation from Chapter 05
    - Diff-based evolution from Chapter 06
    - Checkpointing from Chapter 08

    Note: Process parallelism (Chapter 07) would be added in production.
    Here we use a sequential loop for clarity.
    """

    def __init__(self, config=None):
        self.config = config or Config()

        # Set random seed for reproducibility
        if self.config.random_seed is not None:
            random.seed(self.config.random_seed)

        # Early stopping monitor
        self.early_stopping = EarlyStopping(
            patience=self.config.early_stopping_patience,
            threshold=self.config.convergence_threshold,
            metric=self.config.early_stopping_metric,
        )

        # Statistics tracking
        self.stats = {
            "total_iterations": 0,
            "successful_mutations": 0,
            "failed_mutations": 0,
            "improvements": 0,
            "total_time": 0,
            "best_score_history": [],
        }

    def run(self, initial_code, evaluate_fn,
            max_iterations=None, target_score=None,
            checkpoint_dir=None, resume_from=None):
        """
        Run the complete evolution process.

        Args:
            initial_code: Starting program code
            evaluate_fn: Function that scores a program -> dict of metrics
            max_iterations: Override config max_iterations
            target_score: Stop when this score is reached
            checkpoint_dir: Directory for saving checkpoints
            resume_from: Path to checkpoint to resume from

        Returns:
            dict with 'best_code', 'best_score', 'metrics', 'stats'
        """
        max_iter = max_iterations or self.config.max_iterations
        start_time = time.time()

        # --- Phase 1: Initialize database ---
        # (Using simplified versions from our chapters)
        programs = {}  # id -> {code, metrics, generation, parent_id}
        best_program = None
        best_score = float('-inf')

        # --- Phase 2: Evaluate and store initial program ---
        initial_id = str(uuid.uuid4())[:8]
        initial_metrics = evaluate_fn(initial_code)
        initial_score = initial_metrics.get("combined_score",
            sum(v for v in initial_metrics.values()
                if isinstance(v, (int, float))) / max(1, len(initial_metrics)))

        programs[initial_id] = {
            "code": initial_code,
            "metrics": initial_metrics,
            "score": initial_score,
            "generation": 0,
            "parent_id": None,
            "iteration": 0,
        }
        best_program = programs[initial_id]
        best_score = initial_score

        print(f"Initial program score: {initial_score:.4f}")
        print(f"Starting evolution for up to {max_iter} iterations...")
        print("-" * 60)

        # --- Phase 3: Evolution loop ---
        for iteration in range(1, max_iter + 1):
            self.stats["total_iterations"] = iteration

            # 3a. Select parent (tournament selection from best programs)
            sorted_progs = sorted(programs.values(),
                                  key=lambda p: p["score"], reverse=True)
            # Pick from top programs with some randomness
            top_k = max(1, len(sorted_progs) // 3)
            parent = random.choice(sorted_progs[:top_k])

            # 3b. Mutate (simulate LLM mutation with random changes)
            child_code = self._mutate(parent["code"])

            # 3c. Evaluate child
            try:
                child_metrics = evaluate_fn(child_code)
                child_score = child_metrics.get("combined_score",
                    sum(v for v in child_metrics.values()
                        if isinstance(v, (int, float))) / max(1, len(child_metrics)))
                self.stats["successful_mutations"] += 1
            except Exception:
                self.stats["failed_mutations"] += 1
                continue

            # 3d. Store in database
            child_id = str(uuid.uuid4())[:8]
            programs[child_id] = {
                "code": child_code,
                "metrics": child_metrics,
                "score": child_score,
                "generation": parent["generation"] + 1,
                "parent_id": None,  # simplified
                "iteration": iteration,
            }

            # 3e. Track best
            if child_score > best_score:
                improvement = child_score - best_score
                best_score = child_score
                best_program = programs[child_id]
                self.stats["improvements"] += 1
                print(f"  Iter {iteration}: NEW BEST score={best_score:.4f} "
                      f"(+{improvement:.4f})")

            self.stats["best_score_history"].append(best_score)

            # 3f. Checkpoint
            if checkpoint_dir and iteration % self.config.checkpoint_interval == 0:
                self._save_checkpoint(checkpoint_dir, iteration,
                                     best_program, programs)
                print(f"  Iter {iteration}: Checkpoint saved")

            # 3g. Early stopping check
            should_stop, reason = self.early_stopping.check(
                child_metrics, target_score=target_score
            )
            if should_stop:
                print(f"  EARLY STOPPING at iter {iteration}: {reason}")
                break

            # 3h. Progress report every 20% of iterations
            if iteration % max(1, max_iter // 5) == 0:
                elapsed = time.time() - start_time
                print(f"  Iter {iteration}/{max_iter}: best={best_score:.4f}, "
                      f"programs={len(programs)}, time={elapsed:.1f}s")

        # --- Phase 4: Finalize ---
        self.stats["total_time"] = time.time() - start_time

        print("-" * 60)
        print(f"Evolution complete!")
        print(f"  Best score: {best_score:.4f}")
        print(f"  Total iterations: {self.stats['total_iterations']}")
        print(f"  Improvements found: {self.stats['improvements']}")
        print(f"  Time elapsed: {self.stats['total_time']:.1f}s")

        return {
            "best_code": best_program["code"],
            "best_score": best_score,
            "best_metrics": best_program["metrics"],
            "stats": self.stats,
        }

    def _mutate(self, code):
        """
        Simulate LLM-driven mutation.

        In production, this would:
        1. Build a prompt with parent code + evolution history
        2. Call LLM ensemble to generate a diff
        3. Apply the diff to produce child code

        Here we simulate with random perturbations.
        """
        lines = code.split("\n")
        mutation_type = random.choice(["modify", "swap", "insert"])

        if mutation_type == "modify" and lines:
            idx = random.randint(0, len(lines) - 1)
            # Simulate a small change
            lines[idx] = lines[idx] + "  # mutated"
        elif mutation_type == "swap" and len(lines) > 1:
            i, j = random.sample(range(len(lines)), 2)
            lines[i], lines[j] = lines[j], lines[i]
        elif mutation_type == "insert":
            idx = random.randint(0, len(lines))
            lines.insert(idx, "# evolved line")

        return "\n".join(lines)

    def _save_checkpoint(self, checkpoint_dir, iteration, best_program, programs):
        """Save evolution state to checkpoint"""
        # In production, this serializes the full database
        # Here we just track the checkpoint event
        os.makedirs(checkpoint_dir, exist_ok=True)
        path = os.path.join(checkpoint_dir, f"checkpoint_{iteration}")
        os.makedirs(path, exist_ok=True)
        # Save best program
        with open(os.path.join(path, "best_program.py"), "w") as f:
            f.write(best_program["code"])

print("OpenEvolveController defined successfully!")

## 9.5 Running the Complete System

Let’s put it all together! We’ll evolve a sorting algorithm using our complete controller, just like in previous chapters but now with everything integrated.

In [ ]:
"""Complete evolution demo with our integrated controller"""

import random
import math

# --- Our sorting evaluator (from Chapter 05) ---
def evaluate_sorting(code):
    """
    Evaluate a sorting implementation.
    Returns metrics dict with combined_score.
    """
    test_cases = [
        [3, 1, 2],
        [5, 3, 8, 1, 2],
        [1],
        [],
        [2, 2, 1, 1, 3, 3],
        list(range(10, 0, -1)),
    ]

    correct = 0
    total = len(test_cases)
    total_ops = 0

    for tc in test_cases:
        arr = tc.copy()
        expected = sorted(arr)
        # Simulate running the code on the test case
        # (In production, we'd execute the actual code)
        result = sorted(arr)  # Placeholder

        if result == expected:
            correct += 1
        total_ops += len(arr) * math.log2(max(len(arr), 2))

    correctness = correct / total
    # Simulate efficiency scoring based on code characteristics
    efficiency = max(0, 1.0 - len(code) / 5000)

    combined = 0.6 * correctness + 0.4 * efficiency

    return {
        "combined_score": combined,
        "correctness": correctness,
        "efficiency": efficiency,
        "tests_passed": correct,
        "total_tests": total,
    }

# --- Initial program ---
initial_code = '''def sort(arr):
    """Simple bubble sort"""
    n = len(arr)
    for i in range(n):
        for j in range(0, n - i - 1):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
    return arr
'''

# --- Run evolution! ---
random.seed(42)

controller = OpenEvolveController(Config.from_dict({
    "max_iterations": 100,
    "checkpoint_interval": 25,
    "early_stopping_patience": 30,
    "convergence_threshold": 0.001,
}))

result = controller.run(
    initial_code=initial_code,
    evaluate_fn=evaluate_sorting,
    max_iterations=100,
    target_score=0.99,  # Stop if we reach 99%
    checkpoint_dir=None,  # Skip checkpoints for this demo
)

print(f"\nFinal result: score={result['best_score']:.4f}")
print(f"Metrics: {result['best_metrics']}")

## 9.6 Visualizing the Evolution Process

Let’s visualize how the best score improved over iterations.

In [ ]:
"""Visualize evolution progress"""

import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Best score over iterations
history = result["stats"]["best_score_history"]
axes[0].plot(range(1, len(history) + 1), history, 'b-', linewidth=1.5, alpha=0.8)
axes[0].fill_between(range(1, len(history) + 1), history, alpha=0.15, color='blue')
axes[0].set_xlabel("Iteration", fontsize=12)
axes[0].set_ylabel("Best Score", fontsize=12)
axes[0].set_title("Evolution Progress", fontsize=14)
axes[0].grid(True, alpha=0.3)

# Mark improvements
improvements = []
prev = history[0]
for i, score in enumerate(history):
    if score > prev:
        improvements.append((i + 1, score))
        prev = score

if improvements:
    imp_x, imp_y = zip(*improvements)
    axes[0].scatter(imp_x, imp_y, color='red', s=50, zorder=5, label='Improvements')
    axes[0].legend()

# Plot 2: Statistics summary
stats = result["stats"]
labels = ["Successful\nMutations", "Failed\nMutations", "Improvements"]
values = [stats["successful_mutations"], stats["failed_mutations"], stats["improvements"]]
colors = ['#2ecc71', '#e74c3c', '#3498db']

bars = axes[1].bar(labels, values, color=colors, edgecolor='white', linewidth=2)
axes[1].set_title("Evolution Statistics", fontsize=14)
axes[1].set_ylabel("Count", fontsize=12)

# Add value labels on bars
for bar, val in zip(bars, values):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.5,
                str(val), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.savefig("evolution_complete.png", dpi=150, bbox_inches='tight')
plt.show()
print("Evolution progress visualization saved!")

## 9.7 High-Level API for Easy Usage

OpenEvolve provides a **high-level API** so users don’t need to understand the internal architecture. There are three convenience functions:

| Function | Use Case | Input |
|----------|----------|-------|
| `run_evolution()` | General purpose | Program file + evaluator file |
| `evolve_function()` | Optimize a function | Function + test cases |
| `evolve_code()` | Custom evolution | Code string + evaluator callable |

### Design Principle

$$\text{Ease of Use} = \frac{1}{\text{Lines of Boilerplate}}$$

**Concrete Example:**
> Without API: 50+ lines to set up config, controller, evaluator, database, run loop
> With API: 3 lines: `from openevolve import evolve_function; result = evolve_function(my_func, tests)`
>
> **Life Analogy:** Like using a microwave vs building an oven — the API is the “start” button.

In [ ]:
"""High-level API for OpenEvolve"""

def run_evolution(initial_program, evaluator, config=None,
                  iterations=None, output_dir=None):
    """
    Run evolution with flexible inputs - the main library API.

    This is the simplest way to use OpenEvolve:

        result = run_evolution(
            initial_program="path/to/program.py",
            evaluator="path/to/evaluator.py",
            iterations=100
        )

    Args:
        initial_program: File path or code string
        evaluator: File path or callable
        config: Config object or YAML path
        iterations: Max iterations (overrides config)
        output_dir: Where to save results

    Returns:
        dict with best_code, best_score, metrics, stats
    """
    # Handle config
    if config is None:
        cfg = Config()
    elif isinstance(config, dict):
        cfg = Config.from_dict(config)
    else:
        cfg = config

    # Handle program input
    if os.path.exists(str(initial_program)):
        with open(initial_program, 'r') as f:
            code = f.read()
    else:
        code = str(initial_program)

    # Handle evaluator
    if callable(evaluator):
        eval_fn = evaluator
    elif os.path.exists(str(evaluator)):
        # In production: dynamically load the evaluate() function
        eval_fn = lambda c: {"combined_score": 0.5}  # placeholder
    else:
        raise ValueError("evaluator must be a callable or file path")

    # Create and run controller
    controller = OpenEvolveController(cfg)
    return controller.run(
        initial_code=code,
        evaluate_fn=eval_fn,
        max_iterations=iterations,
        checkpoint_dir=output_dir,
    )


def evolve_function(func, test_cases, iterations=100, **kwargs):
    """
    Evolve a Python function based on test cases.

    The simplest API for optimizing a function:

        def my_sort(arr):
            return sorted(arr)  # starting point

        result = evolve_function(
            my_sort,
            test_cases=[
                ([3, 1, 2], [1, 2, 3]),
                ([5, 2, 8], [2, 5, 8]),
            ],
            iterations=50,
        )
    """
    import inspect
    func_source = inspect.getsource(func)
    func_name = func.__name__

    def evaluator(code):
        # Simplified: in production, write code to temp file,
        # import and run test cases
        correct = 0
        for input_val, expected in test_cases:
            try:
                result = func(input_val.copy() if isinstance(input_val, list) else input_val)
                if result == expected:
                    correct += 1
            except Exception:
                pass
        score = correct / len(test_cases) if test_cases else 0
        return {"combined_score": score, "tests_passed": correct}

    return run_evolution(
        initial_program=func_source,
        evaluator=evaluator,
        iterations=iterations,
        **kwargs,
    )


# Demo the high-level API
print("=== High-Level API Demo ===")

# Using run_evolution with a callable evaluator
result_api = run_evolution(
    initial_program=initial_code,
    evaluator=evaluate_sorting,
    iterations=20,
)
print(f"\nAPI result: score={result_api['best_score']:.4f}")
print(f"Iterations: {result_api['stats']['total_iterations']}")

## 9.8 CLI Entry Point

For command-line usage, OpenEvolve provides a simple entry point:

```bash
# Basic usage
python openevolve-run.py program.py evaluator.py --iterations 1000

# With config file
python openevolve-run.py program.py evaluator.py --config config.yaml

# Resume from checkpoint
python openevolve-run.py program.py evaluator.py --checkpoint output/checkpoints/checkpoint_100

# Override LLM settings
python openevolve-run.py program.py evaluator.py --primary-model gpt-4 --api-base https://api.openai.com/v1
```

### CLI Architecture

```
openevolve-run.py          (Entry point - 3 lines)
    └── openevolve/cli.py      (Argument parsing + async runner)
        └── OpenEvolve()       (Controller from controller.py)
            └── .run()          (Evolution loop)
```

In [ ]:
"""Simplified CLI entry point"""

import argparse

def create_cli():
    """Create the CLI argument parser"""
    parser = argparse.ArgumentParser(
        description="OpenEvolve - Evolutionary coding agent"
    )

    parser.add_argument("initial_program",
        help="Path to the initial program file")
    parser.add_argument("evaluation_file",
        help="Path to the evaluation file with evaluate() function")
    parser.add_argument("--config", "-c",
        help="Path to configuration file (YAML)", default=None)
    parser.add_argument("--output", "-o",
        help="Output directory for results", default=None)
    parser.add_argument("--iterations", "-i",
        help="Maximum number of iterations", type=int, default=None)
    parser.add_argument("--target-score", "-t",
        help="Target score to reach", type=float, default=None)
    parser.add_argument("--checkpoint",
        help="Resume from checkpoint directory", default=None)
    parser.add_argument("--primary-model",
        help="Primary LLM model name", default=None)
    parser.add_argument("--api-base",
        help="Base URL for LLM API", default=None)

    return parser

# Show the CLI help
cli = create_cli()
print("CLI arguments:")
for action in cli._actions:
    if action.option_strings:
        print(f"  {', '.join(action.option_strings):30s} {action.help}")
    elif action.dest != 'help':
        print(f"  {action.dest:30s} {action.help}")

## 9.9 EVOLVE-BLOCK Markers

One important detail: OpenEvolve doesn’t modify the entire program. You mark which sections should be evolved:

```python
import math

# This part stays fixed
INPUT_SIZE = 100

# EVOLVE-BLOCK-START
def my_algorithm(data):
    """This is the part that gets evolved"""
    result = []
    for item in data:
        result.append(item * 2)
    return result
# EVOLVE-BLOCK-END

# This part also stays fixed
if __name__ == "__main__":
    print(my_algorithm(range(INPUT_SIZE)))
```

The LLM only sees and modifies code between the markers. This is crucial for:
1. **Safety** — don’t accidentally break imports or test harnesses
2. **Focus** — LLM concentrates on the algorithm, not boilerplate
3. **Efficiency** — smaller diffs, fewer tokens

In [ ]:
"""EVOLVE-BLOCK marker extraction"""

import re

def extract_evolve_blocks(code):
    """
    Extract code sections marked with EVOLVE-BLOCK-START/END.

    Returns list of (start_line, end_line, block_code) tuples.
    """
    lines = code.split("\n")
    blocks = []
    in_block = False
    block_start = 0
    block_lines = []

    for i, line in enumerate(lines):
        if "EVOLVE-BLOCK-START" in line:
            in_block = True
            block_start = i
            block_lines = []
        elif "EVOLVE-BLOCK-END" in line:
            if in_block:
                blocks.append({
                    "start_line": block_start,
                    "end_line": i,
                    "code": "\n".join(block_lines),
                    "num_lines": len(block_lines),
                })
            in_block = False
        elif in_block:
            block_lines.append(line)

    return blocks


def apply_evolution(original_code, block_index, new_block_code):
    """
    Replace an EVOLVE-BLOCK with new code.
    Keeps everything outside the block unchanged.
    """
    lines = original_code.split("\n")
    blocks = extract_evolve_blocks(original_code)

    if block_index >= len(blocks):
        raise ValueError(f"Block index {block_index} out of range")

    block = blocks[block_index]
    # Replace lines between START and END markers
    new_lines = new_block_code.split("\n")
    result = (
        lines[:block["start_line"] + 1]  # Up to and including START marker
        + new_lines
        + lines[block["end_line"]:]        # From END marker onward
    )

    return "\n".join(result)


# Demo
program = '''import math

INPUT_SIZE = 100

# EVOLVE-BLOCK-START
def my_algorithm(data):
    """Bubble sort - to be evolved"""
    arr = list(data)
    n = len(arr)
    for i in range(n):
        for j in range(n - i - 1):
            if arr[j] > arr[j + 1]:
                arr[j], arr[j + 1] = arr[j + 1], arr[j]
    return arr
# EVOLVE-BLOCK-END

if __name__ == "__main__":
    print(my_algorithm([3, 1, 2]))
'''

blocks = extract_evolve_blocks(program)
print(f"Found {len(blocks)} EVOLVE-BLOCK(s):")
for i, block in enumerate(blocks):
    print(f"  Block {i}: lines {block['start_line']}-{block['end_line']} "
          f"({block['num_lines']} lines)")
    print(f"  Preview: {block['code'][:80]}...")

# Apply an evolved version
evolved_code = '''def my_algorithm(data):
    """Quick sort - evolved!"""
    arr = list(data)
    if len(arr) <= 1:
        return arr
    pivot = arr[len(arr) // 2]
    left = [x for x in arr if x < pivot]
    middle = [x for x in arr if x == pivot]
    right = [x for x in arr if x > pivot]
    return my_algorithm(left) + middle + my_algorithm(right)'''

result = apply_evolution(program, 0, evolved_code)
print(f"\nAfter evolution (first 5 lines of result):")
for line in result.split("\n")[:8]:
    print(f"  {line}")

## 9.10 Complete Architecture Summary

### Component Map

| Our Implementation | OpenEvolve Source | Chapter |
|---|---|---|
| `Config`, `Config.from_yaml()` | `openevolve/config.py` → `Config`, `load_config()` | 09 |
| `OpenEvolveController` | `openevolve/controller.py` → `OpenEvolve` | 09 |
| `run_evolution()` | `openevolve/api.py` → `run_evolution()` | 09 |
| `evolve_function()` | `openevolve/api.py` → `evolve_function()` | 09 |
| `EarlyStopping` | `openevolve/process_parallel.py` (inline logic) | 09 |
| `extract_evolve_blocks()` | `openevolve/prompt_sampler.py` | 09 |
| `SimpleDatabase` (MAP-Elites) | `openevolve/database.py` → `ProgramDatabase` | 02-03 |
| `SmartMutator` | `openevolve/llm/` + `iteration.py` | 04 |
| `CascadeEvaluator` | `openevolve/evaluator.py` → `Evaluator` | 05 |
| `DiffEngine` | `openevolve/diff_utils.py` | 06 |
| `ParallelEvolution` | `openevolve/process_parallel.py` | 07 |
| `CheckpointManager` | `openevolve/database.py` (save/load) | 08 |

### Key Design Decisions in OpenEvolve

| Decision | Why |
|----------|-----|
| Process-based parallelism (not threads) | Avoids Python GIL; true CPU parallelism |
| Database snapshots for workers | Prevents lock contention between processes |
| MAP-Elites + Islands | Diversity preservation + independent exploration |
| Cascade evaluation | Save compute by failing fast on bad programs |
| Diff-based evolution | Smaller changes, fewer tokens, more precise edits |
| EVOLVE-BLOCK markers | Focus LLM on the algorithm, not boilerplate |
| Async controller | LLM API calls are I/O-bound; async maximizes throughput |
| YAML config with CLI overrides | Flexible experimentation without code changes |

### What We Built vs. Production OpenEvolve

| Aspect | Our Tutorial | Production OpenEvolve |
|--------|-------------|----------------------|
| LLM Integration | Simulated random mutations | Real LLM API calls (OpenAI compatible) |
| Parallelism | Sequential (for clarity) | ProcessPoolExecutor with worker processes |
| Config | Simplified dataclasses | Full YAML with dacite deserialization |
| Database | Python dict | MAP-Elites grid with island topology |
| Evaluation | Sync callable | Async with timeout, retry, and cascade |
| Checkpoints | Basic file save | Full database serialization with metadata |

## 9.11 What’s Next?

Congratulations! You’ve built OpenEvolve from scratch, understanding every component:

```
Chapter 00: Why this project matters
Chapter 01: Minimal evolution loop (the seed)
Chapter 02: MAP-Elites diversity (don't put all eggs in one basket)
Chapter 03: Island-based evolution (isolated populations)
Chapter 04: LLM-driven mutation (intelligent code changes)
Chapter 05: Cascade evaluation (fail fast, save compute)
Chapter 06: Diff-based evolution (surgical edits, not rewrites)
Chapter 07: Process parallelism (true multi-core execution)
Chapter 08: Checkpoints (never lose progress)
Chapter 09: Full integration (the complete system)
```

### To Use the Real OpenEvolve

```bash
# Install
pip install -e ".[dev]"

# Run
python openevolve-run.py your_program.py your_evaluator.py \\
    --config config.yaml --iterations 1000

# Resume
python openevolve-run.py your_program.py your_evaluator.py \\
    --checkpoint openevolve_output/checkpoints/checkpoint_500
```

### Ideas for Further Exploration

1. **Custom Evaluator**: Write an evaluator for your own optimization problem
2. **Multi-Language**: Try evolving Rust or R code (just change `file_suffix`)
3. **LLM Ensemble**: Configure multiple models with different weights
4. **Feature Dimensions**: Define custom MAP-Elites features for your domain
5. **Evolution Tracing**: Visualize the evolution tree to understand how solutions develop

### Key Takeaway

> The power of OpenEvolve is not in any single component, but in how they work together:
> **Diversity** (MAP-Elites + Islands) ensures we explore broadly,
> **Intelligence** (LLM mutations) ensures we explore smartly,
> **Efficiency** (Cascade + Diff + Parallel) ensures we explore quickly,
> **Resilience** (Checkpoints + Early Stopping) ensures we don’t waste effort.

Happy evolving! 🧬